# Ingest drivers.json files
1. Read the files using spark dataframe reader API
2. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

name_schema = StructType([
    StructField("givenName", StringType()),
    StructField("familyName", StringType())
])


driver_schema = StructType([
    StructField("driverId", StringType()),
    StructField("name", name_schema),
    StructField("dateOfBirth", DateType()),
    StructField("nationality", StringType()),
    StructField("url", StringType())
])


In [0]:
drivers_df = (
    spark.read
        .format("json")
        .option('header', 'true')
        .option('mode', 'failFast')
        .schema(driver_schema)
        .load(source_file)
)        
       
        
        

In [0]:
display (drivers_df)

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)


In [0]:
display (drivers_final_df)

In [0]:
(
    drivers_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
)

In [0]:
display(spark.read.table(table_name))